<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Explain an alert

**[Explain an alert](https://learning.nextia-ai.com/courses/anomaly-detection/m08/explain-an-alert/)** · Anomaly Detection: Find What Is Unusual · Module 8, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** show which features made an alert's score high, split the re-ranker's score into contributions (the SHAP values of a linear model), and write an alert card that a reviewer can decide on in a minute.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | The re-ranker, from [From unsupervised to semi-supervised](https://learning.nextia-ai.com/courses/anomaly-detection/m08/from-unsupervised-to-semi-supervised/). Contributions of a linear model, from [Explain predictions](https://learning.nextia-ai.com/courses/ml/m07/explain-predictions/) in the machine learning course. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m08/explain-an-alert/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M08-L03-explain-an-alert/explain-an-alert-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, shap 0.54.1. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 shap==0.54.1
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'shap']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One alert, one score

Run the next cell. It builds the candidates, the reviews and the re-ranker of the first lesson of this module. Then it takes the day 2025-10-03 and the alert with the highest re-ranker score.

> **Check.** You should see `P-0229866`, customer `C-01234`, 19.25 dollars in `PT`.

This is the customer of the course's first lesson: eight payments in Portugal in a few minutes. All Sofia sees so far is a score.

In [ ]:
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(scaler.transform(behaviour.loc[reference.index]))
df = evaluation[["payment_id", "ts", "customer_id", "account_type", "amount", "currency", "country",
                 "device_type", "merchant_category"]].copy()
df["day"] = df.ts.dt.floor("D")
df["iso"] = -forest.score_samples(scaler.transform(behaviour.loc[evaluation.index]))
for col in behaviour.columns:
    df[col] = behaviour.loc[evaluation.index, col].to_numpy()
df = df.sort_values(["day", "iso"], ascending=[True, False])
df["rank"] = df.groupby("day").cumcount()
candidates = df[df["rank"] < 200].copy()
candidates["shown"] = candidates["rank"] < 50
# The reviewers' answers, played with the answer key (an assumption: real reviewers make mistakes).
candidates["label"] = candidates.payment_id.isin(compare.planted_payments().payment_id)
labels = candidates[(candidates.day <= "2025-09-28") & candidates.shown]
FEATS = ["iso"] + list(behaviour.columns)
re_scaler = StandardScaler().fit(labels[FEATS])
re_ranker = LogisticRegression(max_iter=2000).fit(re_scaler.transform(labels[FEATS]), labels.label)
day = candidates[candidates.day == "2025-10-03"].copy()
day["re_rank"] = re_ranker.predict_proba(re_scaler.transform(day[FEATS]))[:, 1]
day = day.sort_values("re_rank", ascending=False)
alert = day.iloc[0]
print(alert[["payment_id", "ts", "customer_id", "amount", "currency", "country", "device_type", "re_rank"]].to_string())

## 2. Far from normal: per-feature deviations

The first explanation works for any detector. For each feature, compare the payment with **normal**: the reference months. For a number, give the robust z of Module 2: how many robust sigmas from the median. For a yes/no feature, give the share of normal payments that say yes.

In [ ]:
# How far is each feature from a normal payment? Normal = the reference months (June to August).
normal = behaviour.loc[reference.index]
median = normal.median()
spread = 1.4826 * (normal - median).abs().median()     # the robust spread (MAD) of Module 2
yes_no = ["new_country", "new_device", "night", "is_business"]

def deviations(row):
    out = {}
    for f in behaviour.columns:
        if f in yes_no:
            out[f] = f"{'yes' if row[f] else 'no'} ({normal[f].mean():.1%} of normal payments say yes)"
        elif spread[f] > 0:
            out[f] = f"{row[f]:.2f} (normal {median[f]:.2f}, {(row[f] - median[f]) / spread[f]:+.1f} robust sigma)"
        else:
            out[f] = f"{row[f]:.0f} (normal {median[f]:.0f}; most payments have {median[f]:.0f})"
    return pd.Series(out)

deviations(alert)

> **Check.** `new_country` says `yes (1.1% of normal payments say yes)`.

Some numbers have a spread of 0: most normal payments have the same value (1 payment in the last hour). A robust z would divide by 0, so the cell writes the value and the normal value instead.

## 3. What moved the re-ranker's score: contributions

For a linear model, the score splits exactly into parts. The **contribution** of a feature is its weight times the payment's scaled value. The intercept plus all contributions is the sum, and the logistic function turns the sum into the score. Run the next cell.

In [ ]:
def contributions(row):
    x = re_scaler.transform(row[FEATS].to_frame().T.astype(float))[0]
    return pd.DataFrame({"value": row[FEATS].astype(float).values, "scaled": x,
                         "weight": re_ranker.coef_[0], "contribution": re_ranker.coef_[0] * x}, index=FEATS)

table = contributions(alert)
total = re_ranker.intercept_[0] + table.contribution.sum()
print(table.sort_values("contribution", ascending=False).round(2).to_string())
print(f"intercept {re_ranker.intercept_[0]:.2f} + contributions {table.contribution.sum():.2f} = {total:.2f}")
print(f"score from the sum: {1 / (1 + np.exp(-total)):.4f} | predict_proba: {alert.re_rank:.4f}")

> **Check.** You should see `intercept 0.17 + contributions 8.34 = 8.50` and `score from the sum: 0.9998 | predict_proba: 0.9998`.

## 4. The same numbers, from SHAP

**SHAP values** split any model's score into one part per feature. For a linear model with independent features, the SHAP value of a feature is exactly its contribution, measured from the average of the background data. Here the background is the 1,400 reviewed alerts, whose scaled average is 0. So the SHAP values and the contributions are the same numbers. Run the next cell.

> **Check.** You should see `base value 0.17` and a largest difference of about 0 (for example `1.8e-15`).

In [ ]:
import shap
background = re_scaler.transform(labels[FEATS])
explainer = shap.LinearExplainer(re_ranker, shap.maskers.Independent(background, max_samples=len(background)))
values = explainer.shap_values(re_scaler.transform(alert[FEATS].to_frame().T.astype(float)))[0]
print(f"base value {explainer.expected_value:.2f}")
print(f"largest difference from the contributions: {np.abs(values - table.contribution.values).max():.1e}")

By default, `shap` keeps only 100 rows of the background. The values then move a little. The cell uses all 1,400.

## 5. Write the card

A reviewer does not need nine numbers. She needs the payment, the customer's normal, and the top reasons in words. Run the next cell: it writes a card with the three largest positive contributions.

> **Check.** The first line starts with `P-0229866  C-01234  2025-10-03 02:21  19`.

In [ ]:
def card(row, n=3):
    t = contributions(row)
    top = t[t.contribution > 0].sort_values("contribution", ascending=False).head(n)
    usual = row.amount / np.exp(row.log_amount_vs_usual)
    words = {
        "iso": "unusual overall (detector score {:.2f})".format(row.iso),
        "new_country": f"first payment ever from {row.country}",
        "new_device": "a device this customer never used",
        "night": f"at {row.ts:%H:%M}, at night",
        "device_payments_last_hour": f"this device made {row.device_payments_last_hour:.0f} payments in the last hour",
        "payments_last_hour": f"{row.payments_last_hour:.0f} payments by this customer in the last hour",
        "log_amount_vs_usual": f"{row.amount:.2f} dollars against a usual {usual:.2f}",
        "log_minutes_since_last": "a short time since the last payment",
        "is_business": "a personal account (bulk orders are business)",
    }
    lines = [f"{row.payment_id}  {row.customer_id}  {row.ts:%Y-%m-%d %H:%M}  {row.amount:.2f} dollars "
             f"({row.currency}, {row.country}, {row.device_type})",
             f"Re-ranker score {row.re_rank:.3f}; usual amount {usual:.2f} dollars"]
    lines += [f"  - {words[f]}" for f in top.index]
    return "\n".join(lines)

print(card(alert))

Read the reasons with care. "A personal account" pushes the score up: it is true about the **model** (bulk orders of business accounts were mostly "not"), but it does not help Sofia decide. A good card keeps the reasons that compare the payment with the customer's own normal.

## 6. Check yourself

**Your turn 1.** On the same day, the payment `P-0230910` is also in the re-ranked top 50. Which feature has the largest contribution to its score? Write its name as `largest_push`.

In [ ]:
second = day[day.payment_id == "P-0230910"].iloc[0]
largest_push = contributions(second).contribution.idxmax()

In [ ]:
expect_hash('largest_push', largest_push, 'f81976246fed894a')

**Your turn 2.** Compute the score of `P-0230910` from the intercept and its contributions, with the logistic function. Write it as `score_from_sum`, rounded to 3 decimals.

In [ ]:
t2 = contributions(second)
score_from_sum = round(float(1 / (1 + np.exp(-(re_ranker.intercept_[0] + t2.contribution.sum())))), 3)

In [ ]:
expect('score_from_sum', score_from_sum, 0.387, tolerance=0.001)

In the solution: the largest push is `is_business`, and the score is 0.387. Print `card(second)` and decide, as Sofia would: real or not? The lesson discusses this card.

## Recap

- Per-feature deviations compare an alert with normal, for any detector.
- For the re-ranker, a linear model, the score splits exactly into contributions; SHAP values give the same numbers.
- A card shows the payment, the customer's normal and the top reasons in words. A contribution is the model's view, not a cause.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [From score to alert](https://learning.nextia-ai.com/courses/anomaly-detection/m09/from-score-to-alert/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m08/explain-an-alert/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.